## Step 1 — Data Loading & Overview

We load three CSV files from the Tracy IoT contact tracing system deployed across Nigeria between October 2023 and June 2024. Each file captures a different phase of the system:

- **contact_tracing.csv** (119,338 rows, 88 users): Bluetooth contact detection events — who was near whom, where, and when. October to December 2023.
- **mobility.csv** (50,737 rows, 79 users): Location pings with pre-computed exposure scores per user. January to March 2024.
- **vitals.csv** (2,307 rows, 21 devices): Health readings from stationary monitoring stations in Akure, Ondo State. May to June 2024.

**Key limitation:** The three files cover different time periods and deployment phases. They cannot be merged on date. Each is analysed independently and findings combined at the interpretation stage.

## Step 2 — Feature Engineering Documentation

All features are computed at the **per-user level** — one row per user_id.

| Feature | Source Columns | Formula / Logic | Why Chosen | What It Captures |
|---|---|---|---|---|
| exposure_score | mobility.csv — exposure_score | Pre-computed by dataset: weighted sum of all detections where very_close=3, close=2, moderate=1, distant=0.5, unscored=0.5 | Already encodes proximity weighting — most comprehensive single risk measure | Total accumulated contact exposure across the full study period |
| daily_exposure | mobility.csv — daily_exposure | Pre-computed: exposure_score / number of active contact days | Normalises for device activity — a user active 4 days with daily_exposure=4,589 is more dangerous than one active 27 days with daily_exposure=379 | Exposure intensity per active day — accounts for varying device usage patterns |
| total_detections | mobility.csv — total_detections | Pre-computed: count of all Bluetooth detections per user | Raw contact volume baseline | How many Bluetooth detection events this user recorded across the study period |
| close_contacts | mobility.csv — close_contacts | Pre-computed: count of detections where proximity is close or very close | Counts only highest-risk events | Number of contacts within transmission-relevant distance |
| active_days | mobility.csv — date | df.groupby('user_id')['date'].nunique() | Measures how long each user was tracked | Number of distinct days the device was active |
| contact_days | mobility.csv — has_contact | df.groupby('user_id')['has_contact'].sum() | Distinguishes days with proximity events from quiet days | Number of days where at least one Bluetooth contact was recorded |
| total_contact_events | contact_tracing.csv — mac | df.groupby('user_id')['mac'].count() | Overall detection volume from raw event log | Total number of Bluetooth detection events including unscored |
| unique_macs | contact_tracing.csv — mac | df.groupby('user_id')['mac'].nunique() | Measures breadth of contact network — a user touching 3,605 unique devices is a super-spreader candidate | Number of distinct Bluetooth devices detected by this user |
| close_very_close_count | contact_tracing.csv — proximity | Count of rows where proximity is 'close' or 'very close', excluding unscored | Absolute volume of highest-risk contacts — complements close_contact_rate | Raw count of contacts within close or very close transmission distance |
| scored_count | contact_tracing.csv — proximity | Count of rows where proximity is not 'unscored' | Denominator for close_contact_rate — ensures rate is not distorted by missing RSSI values | Number of contacts where signal strength was successfully recorded |
| close_contact_rate | contact_tracing.csv — proximity | close_very_close_count / scored_count (0 where scored_count = 0) | Proportion of measurable contacts that were high-risk — U061 had 99.9% close rate despite lower volume | Quality of contacts — what fraction of scored detections were within transmission distance |
| degree_centrality | contact_tracing.csv — user_id, date, geohash | nx.degree_centrality(G) where G is built from pairs of users sharing the same geohash on the same date | Captures network position — a user central in the co-location graph can seed an outbreak across the entire study population | Fraction of all other Tracy users this user physically co-located with |

**Features excluded and why:**

| Feature | Reason for Exclusion |
|---|---|
| total_detections | High correlation with close_very_close_count and exposure_score — multicollinearity |
| total_contact_events | Redundant with unique_macs and close_very_close_count |
| active_days | Operational metric — does not indicate risk level |
| contact_days | Captured more meaningfully by daily_exposure |
| scored_count | Intermediate calculation — not a risk indicator itself |

**Note on unscored contacts:** 61% of contact_tracing.csv rows have rssi = -1 (unscored). These were excluded when computing close_contact_rate to avoid artificially deflating the rate. They are retained in total_contact_events and unique_macs since detection volume and breadth are valid regardless of signal measurement.

## Step 3 Results — Cluster Findings

KMeans produced three distinct user groups:

| Tier | Users | Avg Exposure Score | Avg Daily Exposure | Avg Close Contact Rate |
|---|---|---|---|---|
| Extreme Risk | 1 (U073) | 18,359 | 4,590 | 99.8% |
| High Risk | 8 users | 1,178 | 52.6 | 96.0% |
| Low Risk | 79 users | 461 | 102.9 | 89.0% |

**Critical finding:** U073 formed its own cluster — its exposure score is so extreme that KMeans could not group it with any other user. This is not a modelling error; it reflects a genuine outlier of public health significance.

**Second critical finding:** 6 of the 8 High Risk users have exposure_score of 0 in mobility.csv — they appear only in contact_tracing.csv. They were correctly flagged by unique_macs and close_contact_rate alone. A model using only exposure_score would have labelled these users Low Risk and missed them entirely. This validates the multi-feature approach.

## Step 4 — Anomaly Detection Results

We applied Z-score anomaly detection to daily contact volumes across the 51-day study period (October to December 2023). Days exceeding mean + 2 standard deviations were flagged as anomalous.

- **Mean daily detections:** 2,340
- **Anomaly threshold (mean + 2σ):** 10,332
- **Anomalous days detected:** 2

| Date | Total Detections | Active Users | Close Events | Z-Score | Interpretation |
|---|---|---|---|---|---|
| 2023-10-23 | 22,720 | 9 | 7,975 | 5.10 | Critical — likely mass gathering event |
| 2023-11-05 | 13,438 | 13 | 5,250 | 2.78 | High — possible secondary spread event |

**Epidemiological significance:** The 13-day gap between the two anomalous days falls within the typical incubation period of respiratory infectious diseases (2–14 days). This suggests the November 5th event may represent secondary transmission from the October 23rd gathering.

**Public health action:** Investigate what occurred in the deployment area on these two specific dates. Cross-reference with local market days, religious gatherings, or transport events.

## Step 5 — Vitals Analysis Results

We analysed 2,307 health readings from 21 stationary monitoring devices in Akure, Ondo State (May–June 2024).

**Important limitation:** Devices were stationary — readings reflect people passing through the monitoring location, not specific named individuals. Surface temperature reads 1–2°C below core temperature, so a surface reading of 37.5°C corresponds to a core temperature of approximately 38.5–39°C clinically.

| Finding | Value | Clinical Significance |
|---|---|---|
| Fever readings (temp > 37.5°C) | 137 (5.9%) | Potential febrile illness at monitoring locations |
| Tachycardia readings (HR > 100 BPM) | 133 (5.8%) | Elevated heart rate — multiple possible causes |
| Dual anomaly (fever + tachycardia) | 1 event | Strongest disease signal — SIRS criteria met |

**D020 — Critical monitoring location:**
50.9% of all readings showed elevated surface temperature. Max temperature recorded: 42.0°C. One dual anomaly on May 26, 2024 (38.0°C surface + 105 BPM). This location requires immediate clinical investigation.

**D019 — Persistent tachycardia location:**
114 of 115 readings showed heart rate above 100 BPM. Average heart rate: 117.69 BPM. This could indicate a high-activity or high-stress environment, or a population with underlying cardiac conditions at this monitoring point.

## Limitations & Future Work

**Limitations:**
- No ground truth infection labels exist — clustering and anomaly scores are proxies for risk, not confirmed diagnoses
- 61% of contact events in contact_tracing.csv are unscored (rssi = -1) — proximity cannot be measured for the majority of detections
- The three files cover different time periods and deployment phases — direct cross-file merging is not possible without geographic and temporal matching
- Vitals devices were stationary — readings reflect multiple people passing through, not individual-level health tracking
- Surface temperature reads 1–2°C below core body temperature — the fever threshold should be interpreted conservatively
- 49 of 79 mobility users had zero contact events — their risk scores are zero by default, not by confirmed safety

**Future work:**
- Collect ground truth disease outcome data to enable supervised classification and validate unsupervised risk scores
- Deploy denser vitals coverage so individual-level health tracking becomes possible
- Integrate geohash matching between mobility.csv and vitals.csv to correlate high-contact zones with high-vitals-anomaly zones
- Build a continuous scoring pipeline that rescores all users every 24 hours as new contact data arrives
- Train a supervised classifier using self-engineered labels (top 10% by exposure score as positive class) to predict future high-risk users from early activity patterns